# A week ahead, from every origin

A parts warehouse plans staff a week in advance, so each Wednesday night it needs a forecast of
the orders on each of the next seven days. You will build an autoregressive forecaster in
NumPy, run it forward a week by feeding its own predictions back in, score it with MASE, and
compare it with the seasonal naive forecast at many forecast origins rather than one.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Three years of daily orders: a weekly pattern with quiet weekends, slow growth, and
# noise that carries over from one day to the next. Day 0 is a Monday.
rng = np.random.default_rng(12)
n = 1095
t = np.arange(n)
weekday = np.array([30.0, 34.0, 33.0, 35.0, 31.0, 12.0, 8.0])
noise = np.zeros(n)
for i in range(1, n):
    noise[i] = 0.5 * noise[i - 1] + rng.normal(0, 2.0)
orders = weekday[t % 7] + 0.01 * t + noise
print(f"{n} days, first week {np.round(orders[:7], 1)}")

## Part 1 (coding task): Fit an AR(L) model by ridge regression

Write `fit_ar(series, L, lam)`. Every run of $L + 1$ consecutive values of `series` is one
training row: its first $L$ values, oldest first, are the inputs and its last value is the
target. Return the weights `w` (length $L$, in the same oldest-first order) and the intercept
`b` that minimize

$$
\sum_{\text{rows}} \big(\text{target} - \mathbf{w}^\top \mathbf{x} - b\big)^2 + \lambda \lVert \mathbf{w} \rVert^2 ,
$$

with no penalty on `b`. With $\lambda = 0$ this is ordinary least squares.

In [ ]:
def fit_ar(series, L, lam):
    """Ridge fit of the next value on the previous L values. Returns (w, b), w oldest first."""
    series = np.asarray(series, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
y_ar = np.zeros(40)
y_ar[0], y_ar[1] = 1.0, 4.0
for i in range(2, 40):
    y_ar[i] = 0.8 * y_ar[i - 1] - 0.3 * y_ar[i - 2] + 2        # an exact AR(2), no noise
w, b = fit_ar(y_ar, 2, 0.0)
assert np.shape(w) == (2,), f"w should have length L = 2, got shape {np.shape(w)}"
assert np.allclose(w, [-0.3, 0.8], atol=1e-6), f"w = {w}: on an exact AR(2) with lam = 0 expect [-0.3, 0.8], oldest lag first; check which value is the target of each row"
assert np.isclose(b, 2.0, atol=1e-6), f"b = {b}: expected the intercept 2"
walk = np.random.default_rng(5).normal(size=60).cumsum()
w, b = fit_ar(walk, 3, 4.0)
assert np.allclose(w, [0.00109455, 0.05785241, 0.92453576], atol=1e-6), f"w = {w}: wrong ridge weights; the penalty applies to w only, not to b"
assert np.isclose(b, -0.32803253, atol=1e-6), f"b = {b}: wrong intercept"
print("Part 1 passed")

## Part 2 (coding task): Run the one-step model a week forward

Write `recursive_forecast(w, b, history, H)`. Start from the last $L$ values of `history`
(where $L$ is the length of `w`), predict the next value with $\mathbf{w}^\top \mathbf{x} + b$,
and use that prediction as the newest input for the step after it. Return the $H$
predictions as an array.

In [ ]:
def recursive_forecast(w, b, history, H):
    """H forecasts from one model, each fed back in as the newest input."""
    w = np.asarray(w, dtype=float)
    history = np.asarray(history, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
f = recursive_forecast([-0.3, 0.8], 2.0, [7.0, 5.0, 10.0], 3)
assert np.shape(f) == (3,), f"expected 3 forecasts, got shape {np.shape(f)}"
assert np.isclose(f[0], 8.5), f"first forecast {f[0]}: expected 0.8 * 10 - 0.3 * 5 + 2 = 8.5, from the last L = 2 values of the history"
assert np.allclose(f, [8.5, 5.8, 4.09]), f"forecasts {f}: expected [8.5, 5.8, 4.09]; each prediction becomes the newest input and the oldest input drops out"
w, b = fit_ar(orders[:700], 14, 1.0)
f = recursive_forecast(w, b, orders[:700], 7)
assert np.isclose(f[0], w @ orders[686:700] + b), "the first forecast should use the last 14 days of the history"
print("Part 2 passed")

## Part 3 (coding task): MASE

Write `mase(actual, forecast, train, m)`: the mean absolute error of `forecast` against
`actual`, divided by the mean absolute error that the seasonal naive forecast with season
$m$ makes on `train`, predicting each training value from the one $m$ steps before it.

In [ ]:
def mase(actual, forecast, train, m):
    """MAE of the forecast over the in-sample MAE of seasonal naive with season m."""
    actual, forecast, train = (np.asarray(a, dtype=float) for a in (actual, forecast, train))
    # TODO
    raise NotImplementedError

In [ ]:
train = [1.0, 4.0, 2.0, 8.0, 3.0, 9.0]
value = mase([10.0, 12.0], [11.0, 15.0], train, 2)
assert not np.isclose(value, 2 / 4.4), "this scale uses one-step differences; the seasonal naive forecast compares each value with the one m steps before"
assert not np.isclose(value, 2 / 8.75), "this scale is the mean of second differences, np.diff(train, 2): the second argument of np.diff repeats the difference, it is not a lag"
assert np.isclose(value, 2 / 1.75), f"MASE = {value}: expected MAE 2 over the scale (1 + 4 + 1 + 1) / 4 = 1.75"
assert np.isclose(mase([5.0, 5.0], [5.0, 5.0], train, 2), 0.0), "a perfect forecast has MASE 0"
print("Part 3 passed")

## Part 4 (coding task): Score from every origin

One cut gives one score, and one week can be lucky. Write
`rolling_origin(series, first_origin, H, step, L, lam, m)`. For each forecast origin
$o$ = `first_origin`, `first_origin + step`, and so on, for as long as the $H$ values
`series[o:o + H]` all exist:

- Fit `fit_ar` with `L` and `lam` on `series[:o]` only, everything known at the origin.
- Forecast `series[o:o + H]` with `recursive_forecast` from that same history.
- Forecast it with seasonal naive, `series[o - m:o - m + H]`, which needs $H \le m$.
- Score both with `mase`, using `series[:o]` as the training data.

Return two arrays, the MASE of the AR forecasts and of the seasonal naive forecasts, one
entry per origin.

In [ ]:
def rolling_origin(series, first_origin, H, step, L, lam, m):
    """MASE of recursive AR(L) and of seasonal naive at each origin, refitting every time."""
    series = np.asarray(series, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
ar, sn = rolling_origin(orders[:119], 84, 7, 7, 14, 1.0, 7)
assert len(ar) == len(sn) == 5, f"{len(ar)} origins: expected 5 (84, 91, 98, 105, 112); with 119 values the last origin, 112, forecasts series[112:119], the final seven"
assert np.allclose(sn, [0.65267769, 0.87407322, 0.79594977, 0.99686903, 1.10405037], atol=1e-6), f"seasonal naive MASE {np.round(sn, 4)}: the scale must come from series[:o] at each origin"
assert np.allclose(ar, [0.89931739, 0.89470283, 0.87985836, 0.40714805, 1.00615709], atol=1e-6), f"AR MASE {np.round(ar, 4)}: refit on series[:o] at every origin and use nothing after it"

ar, sn = rolling_origin(orders, 730, 7, 28, 28, 1.0, 7)
print(f"{len(ar)} origins, one every four weeks over the third year")
print(f"mean MASE   recursive AR(28) {ar.mean():.3f}   seasonal naive {sn.mean():.3f}")
print(f"AR better at {np.sum(ar < sn)} of {len(ar)} origins; AR MASE ranges from {ar.min():.2f} to {ar.max():.2f}")
assert ar.mean() < sn.mean(), "on the warehouse data the AR model should beat seasonal naive on average"
print("Part 4 passed")

## Solution

### Part 1: Fit an AR(L) model by ridge regression

In [ ]:
def fit_ar(series, L, lam):
    """Ridge fit of the next value on the previous L values. Returns (w, b), w oldest first."""
    series = np.asarray(series, dtype=float)
    win = np.lib.stride_tricks.sliding_window_view(series, L + 1)
    X, y = win[:, :L], win[:, L]
    x_mean, y_mean = X.mean(axis=0), y.mean()
    Xc, yc = X - x_mean, y - y_mean
    gram = Xc.T @ Xc
    w = np.linalg.solve(gram + lam * np.eye(L), Xc.T @ yc)
    return w, y_mean - x_mean @ w

**Why it works.** Centering the inputs and the targets removes the intercept from the problem: the
penalized least-squares weights of the centered data solve
$(X_c^\top X_c + \lambda I)\,\mathbf{w} = X_c^\top \mathbf{y}_c$, and the intercept that goes
with them is $\bar{y} - \bar{\mathbf{x}}^\top \mathbf{w}$, which is exactly what an
unpenalized $b$ chooses. Adding a column of ones and leaving its diagonal entry out of the
penalty gives the same answer. `sliding_window_view(series, L + 1)` makes every row at once,
and its last column is the value that follows the $L$ inputs.

### Part 2: Run the one-step model a week forward

In [ ]:
def recursive_forecast(w, b, history, H):
    """H forecasts from one model, each fed back in as the newest input."""
    w = np.asarray(w, dtype=float)
    history = np.asarray(history, dtype=float)
    window = history[-len(w):].copy()               # oldest first, like the training rows
    out = np.empty(H)
    for h in range(H):
        out[h] = w @ window + b
        window = np.append(window[1:], out[h])      # drop the oldest, add the newest
    return out

**Why it works.** The window always holds the $L$ most recent values, real or predicted, in the
same oldest-first order the weights were fitted on. After the first step the newest input is
a prediction, so any error in it is carried into the next step: that is the price of using one
model for every horizon. A forecast that reused the same real window $H$ times would give the
same number for every day of the week.

### Part 3: MASE

In [ ]:
def mase(actual, forecast, train, m):
    """MAE of the forecast over the in-sample MAE of seasonal naive with season m."""
    actual, forecast, train = (np.asarray(a, dtype=float) for a in (actual, forecast, train))
    scale = np.mean(np.abs(train[m:] - train[:-m]))      # y_t against y_(t - m)
    return np.mean(np.abs(actual - forecast)) / scale

**Why it works.** The scale is a fixed property of the training data: how far, on average, each
value sits from the value one season earlier. Dividing by it makes the score unit-free, and a
value below 1 means the forecast misses by less than seasonal naive did in sample. The lag-$m$
difference is `train[m:] - train[:-m]`. `np.diff(train, m)` differences $m$ times in a row,
which for $m = 7$ is a seventh-order difference and a far larger number.

### Part 4: Score from every origin

In [ ]:
def rolling_origin(series, first_origin, H, step, L, lam, m):
    """MASE of recursive AR(L) and of seasonal naive at each origin, refitting every time."""
    series = np.asarray(series, dtype=float)
    ar, seasonal = [], []
    for o in range(first_origin, len(series) - H + 1, step):
        train, actual = series[:o], series[o:o + H]
        w, b = fit_ar(train, L, lam)
        ar.append(mase(actual, recursive_forecast(w, b, train, H), train, m))
        seasonal.append(mase(actual, series[o - m:o - m + H], train, m))
    return np.array(ar), np.array(seasonal)

**Why it works.** Each origin repeats exactly what will happen in use: everything before $o$ is
known, the model is fitted to it and nothing else, and the next $H$ days are scored. The
training set grows by `step` values from one origin to the next, which is the rolling forecasting
origin from the lesson. The loop runs while $o + H \le n$, so the last week of the series is
used when it fits exactly.

On the warehouse data the AR model averages a MASE of about 0.87 against 0.92 for seasonal
naive, so it beats the baseline on average, but by little, and at only 6 of the 13 origins. Its own score
ranges from about 0.49 to 1.32 between origins. A single cut could have reported either end of
that range, which is the reason to average over many origins before choosing a model.